In [2]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")

Dataset berhasil dibuat: 2500 baris


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara-Regresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)
print("Total baris:", df.count())

26/11/18 21:34:27 WARN Utils: Your hostname, raam resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/18 21:34:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 21:34:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)



+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



[Stage 3:>                                                          (0 + 1) / 1]

Total baris: 2500


In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Fit StringIndexer untuk masing-masing kolom kategorikal
model_idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df)
model_idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df)
model_idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df)

# 2. Transformasikan data menggunakan model StringIndexer yang telah di-fit
df_indexed = model_idx_menu.transform(df)
df_indexed = model_idx_cabang.transform(df_indexed)
df_indexed = model_idx_waktu.transform(df_indexed)

# 3. Gabungkan seluruh fitur ke dalam satu kolom vector "features"
assembler_utama = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_final = assembler_utama.transform(df_indexed)
df_final.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "features", "total_penjualan").show(5, truncate=False)

[Stage 15:>                                                         (0 + 1) / 1]

+--------------+----------+-----------+---+------------+-------------------------+------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|features                 |total_penjualan   |
+--------------+----------+-----------+---+------------+-------------------------+------------------+
|Kentang Goreng|Semarang  |Malam      |1  |30000       |[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|Kentang Goreng|Yogyakarta|Sore       |3  |20000       |[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|Kentang Goreng|Semarang  |Siang      |5  |22000       |[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|Kentang Goreng|Yogyakarta|Pagi       |2  |20000       |[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|Roti Bakar    |Magelang  |Malam      |2  |30000       |[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+--------------+----------+-----------+---+------------+-------------------------+------------------+
only showing top 5 rows



In [5]:
from pyspark.ml.regression import LinearRegression

train_df, test_df = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training : {train_df.count()}")
print(f"Jumlah data testing  : {test_df.count()}")

lr_utama = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr_utama.fit(train_df)

print("Model utama berhasil dilatih!")
print("Koefisien fitur :", model_utama.coefficients)
print("Intercept       :", model_utama.intercept)

Jumlah data training : 1839


Jumlah data testing  : 661


Model utama berhasil dilatih!
Koefisien fitur : [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept       : -66903.17766485889


In [6]:
from pyspark.ml.regression import LinearRegression

train_df, test_df = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training : {train_df.count()}")
print(f"Jumlah data testing  : {test_df.count()}")

lr_utama = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr_utama.fit(train_df)

print("Model utama berhasil dilatih!")
print("Koefisien fitur :", model_utama.coefficients)
print("Intercept       :", model_utama.intercept)

Jumlah data training : 1839


Jumlah data testing  : 661


[Stage 31:>                                                         (0 + 1) / 1]

Model utama berhasil dilatih!
Koefisien fitur : [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept       : -66903.17766485889


In [7]:
from pyspark.ml.regression import LinearRegression

train_df, test_df = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training : {train_df.count()}")
print(f"Jumlah data testing  : {test_df.count()}")

lr_utama = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_utama = lr_utama.fit(train_df)

print("Model utama berhasil dilatih!")
print("Koefisien fitur :", model_utama.coefficients)
print("Intercept       :", model_utama.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661


[Stage 39:>                                                         (0 + 1) / 1]

Model utama berhasil dilatih!
Koefisien fitur : [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept       : -66903.17766485889


In [9]:
skenario_baru = [
    ("Kopi Susu", "Magelang", "Pagi", 3, 20000),
    ("Roti Bakar", "Yogyakarta", "Sore", 2, 25000),
    ("Cappuccino", "Semarang", "Malam", 5, 30000)
]

kolom_skenario = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_skenario = spark.createDataFrame(skenario_baru, schema=kolom_skenario)

# Terapkan model StringIndexer yang SAMA dari Bagian A (tanpa fit ulang)
df_skenario_idx = model_idx_menu.transform(df_skenario)
df_skenario_idx = model_idx_cabang.transform(df_skenario_idx)
df_skenario_idx = model_idx_waktu.transform(df_skenario_idx)

# Terapkan VectorAssembler yang SAMA dari Bagian A
df_skenario_final = assembler_utama.transform(df_skenario_idx)

# Prediksi menggunakan model utama
hasil_prediksi = model_utama.transform(df_skenario_final)
hasil_prediksi.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show(truncate=False)

# Menutup SparkSession setelah seluruh bagian selesai
spark.stop()
print("SparkSession ditutup.")

AttributeError: 'NoneType' object has no attribute 'sc'